In [3]:
import sqlite3, json, pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

base = Path(".")

db_path = base / "download"
json_path = base / "download (1)"
html_path = base / "download (2)"


# Load the three provided sources

con = sqlite3.connect(db_path)

members = pd.read_sql("SELECT * FROM members", con)
books_db = pd.read_sql("SELECT * FROM books", con)
checkouts = pd.read_sql("SELECT * FROM checkouts", con)

with open(json_path, encoding="utf-8") as f:
    catalog = pd.DataFrame(json.load(f))

with open(html_path, encoding="utf-8") as f:
    soup = BeautifulSoup(f.read(), "html.parser")

rows = []
table = soup.find("table")

for tr in table.find_all("tr")[1:]:
    cells = [td.get_text(strip=True) for td in tr.find_all("td")]
    rows.append({
        "member_id": int(cells[0]),
        "book_id": int(cells[1]),
        "checkout_date": cells[2]
    })

kickoff = pd.DataFrame(rows)

print("Database and source files loaded successfully!")
print("Members:", len(members))
print("Books:", len(books_db))
print("Checkouts:", len(checkouts))
print("Reading Kickoff:", len(kickoff))

Database and source files loaded successfully!
Members: 80
Books: 32
Checkouts: 391
Reading Kickoff: 26


In [4]:

# Task 1 / SQL answers


queries = {
    "Question 1 — Checkouts per Member": """
SELECT
    m.member_id,
    m.first_name,
    m.last_name,
    COUNT(c.checkout_id) AS checkout_count
FROM members AS m
LEFT JOIN checkouts AS c
    ON m.member_id = c.member_id
GROUP BY
    m.member_id,
    m.first_name,
    m.last_name
ORDER BY m.member_id;
""",

    "Question 2 — Books by Author Pattern": """
SELECT
    book_id,
    title,
    author
FROM books
WHERE author LIKE 'S%'
ORDER BY book_id;
""",

    "Question 3 — Most Frequently Borrowed Books": """
SELECT
    b.book_id,
    b.title,
    COUNT(c.checkout_id) AS checkout_count
FROM books AS b
JOIN checkouts AS c
    ON b.book_id = c.book_id
GROUP BY
    b.book_id,
    b.title
ORDER BY
    checkout_count DESC,
    b.book_id
LIMIT 5;
""",

    "Question 4 — Members with the Most Borrowed Books": """
SELECT
    m.member_id,
    m.first_name,
    m.last_name,
    COUNT(c.checkout_id) AS book_count
FROM members AS m
JOIN checkouts AS c
    ON m.member_id = c.member_id
GROUP BY
    m.member_id,
    m.first_name,
    m.last_name
ORDER BY
    book_count DESC,
    m.member_id
LIMIT 10;
""",

    "Question 5 — Neighborhood Checkouts, Second Set of 10": """
SELECT
    c.checkout_id,
    c.member_id,
    c.book_id,
    c.checkout_date,
    c.return_date
FROM checkouts AS c
JOIN members AS m
    ON c.member_id = m.member_id
WHERE m.neighborhood = 'Maadi'
ORDER BY
    c.checkout_date DESC,
    c.checkout_id DESC
LIMIT 10 OFFSET 10;
"""
}

results = {}

for title, query in queries.items():
    results[title] = pd.read_sql(query, con)

print("All 5 SQL queries executed successfully!")

for title, result in results.items():
    print("\n" + title)
    print(result)

All 5 SQL queries executed successfully!

Question 1 — Checkouts per Member
    member_id first_name last_name  checkout_count
0        1001      Salma   Ibrahim               1
1        1002      Fares     Saleh               2
2        1003     Bassel    Hegazy               9
3        1004      Fares     Wahba               0
4        1005    Youssef     Halim               3
..        ...        ...       ...             ...
75       1076       Dina     Wahba               7
76       1077       Lina    Rashad               6
77       1078     Habiba     Osman               0
78       1079       Rana     Osman              10
79       1080     Bassel     Wahba               2

[80 rows x 4 columns]

Question 2 — Books by Author Pattern
   book_id                   title        author
0      529  Riddles of the Red Sea  Sara Tantawy
1      530       The Sandstone Key  Sara Tantawy
2      531   Voices in the Library   Samir Zohdy
3      532       The Last Bookmark   Samir Zohdy

Quest

In [5]:
# Stage 1: Members + Checkouts
# Python only
member_checkout_counts = (
    checkouts.groupby("member_id")
    .size()
    .rename("member_total_book_count")
    .reset_index()
)

stage1 = checkouts.merge(
    members,
    on="member_id",
    how="left",
    validate="many_to_one"
).merge(
    member_checkout_counts,
    on="member_id",
    how="left",
    validate="many_to_one"
)

print("Stage 1 completed.")
print("Stage 1 rows:", len(stage1))
print("Original checkout rows:", len(checkouts))

Stage 1 completed.
Stage 1 rows: 391
Original checkout rows: 391


In [6]:
# Stage 2: Add Book Catalog
stage2 = stage1.merge(
    books_db,
    on="book_id",
    how="left",
    validate="many_to_one"
)

stage2 = stage2.merge(
    catalog,
    on="book_id",
    how="left",
    validate="many_to_one"
)

print("Stage 2 completed.")
print("Stage 2 rows:", len(stage2))
print("Stage 1 rows:", len(stage1))

Stage 2 completed.
Stage 2 rows: 391
Stage 1 rows: 391


In [7]:
# Stage 3: Add Reading Kickoff
max_checkout_id = int(
    checkouts["checkout_id"].max()
)

kickoff = kickoff.copy()

kickoff["checkout_id"] = range(
    max_checkout_id + 1,
    max_checkout_id + 1 + len(kickoff)
)

kickoff["return_date"] = pd.NA
kickoff["source"] = "Reading Kickoff"

stage2["source"] = "Library Database"

# Bring kickoff records to the same checkout-oriented schema.
kickoff_full = kickoff.merge(
    members,
    on="member_id",
    how="left",
    validate="many_to_one"
).merge(
    member_checkout_counts,
    on="member_id",
    how="left"
)

kickoff_full = kickoff_full.merge(
    books_db,
    on="book_id",
    how="left",
    validate="many_to_one"
).merge(
    catalog,
    on="book_id",
    how="left",
    validate="many_to_one"
)

# Database records get 0 for members who somehow have no count.
stage2["member_total_book_count"] = (
    stage2["member_total_book_count"]
    .fillna(0)
    .astype(int)
)

# Kickoff member IDs not in the registered members table remain unmatched.
combined = pd.concat(
    [stage2, kickoff_full],
    ignore_index=True,
    sort=False
)

print("Stage 3 completed.")
print("Reading Kickoff records:", len(kickoff))
print("Final combined rows:", len(combined))

Stage 3 completed.
Reading Kickoff records: 26
Final combined rows: 417


In [8]:

# Organize Combined Dataset


preferred = [
    "checkout_id",
    "member_id",
    "book_id",
    "checkout_date",
    "return_date",
    "first_name",
    "last_name",
    "grade",
    "neighborhood",
    "membership_status",
    "join_date",
    "member_total_book_count",
    "title",
    "author",
    "genre",
    "pages",
    "publication_year",
    "publisher",
    "source"
]

combined = combined[
    [c for c in preferred if c in combined.columns]
]

print("Combined dataset columns:")
print(combined.columns.tolist())

print("\nCombined dataset shape:")
print(combined.shape)

display(combined.head())

Combined dataset columns:
['checkout_id', 'member_id', 'book_id', 'checkout_date', 'return_date', 'first_name', 'last_name', 'grade', 'neighborhood', 'membership_status', 'join_date', 'member_total_book_count', 'title', 'author', 'genre', 'pages', 'publication_year', 'publisher', 'source']

Combined dataset shape:
(417, 19)


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,member_total_book_count,title,author,genre,pages,publication_year,publisher,source
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,16.0,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House,Library Database
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,14.0,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,Library Database
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,5.0,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books,Library Database
3,9129,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,6.0,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,Library Database
4,9370,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27,10.0,Winter in Alexandria,Farida Anwar,Historical,117,2016.0,Nile Press,Library Database


In [9]:

# Save Task 1 files


sql_path = base / "task1_sql_answers.txt"
csv_path = base / "task1_combined_data.csv"

with open(
    sql_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "WEEK 20 PROJECT — TASK 1 SQL ANSWERS\n"
    )

    f.write(
        "=" * 70 + "\n\n"
    )

    f.write(
        "Question 2 chosen author pattern: "
        "author name starts with 'S'.\n"
    )

    f.write(
        "Question 5 chosen neighborhood: Maadi.\n\n"
    )

    for title, query in queries.items():

        f.write(
            title + "\n"
        )

        f.write(
            "-" * len(title) + "\n"
        )

        f.write(
            "SQL:\n"
        )

        f.write(
            query.strip() + "\n\n"
        )

        f.write(
            "Result:\n"
        )

        f.write(
            results[title].to_string(index=False)
            + "\n\n"
        )

    f.write(
        "WEB PAGE VS API REFLECTION\n"
    )

    f.write(
        "==========================\n"
    )

    f.write(
        "The Reading Kickoff source was provided as an HTML web page "
        "rather than as an API. The HTML page is designed primarily "
        "for a person to read in a browser, so the checkout information "
        "had to be extracted from the table and then reshaped to match "
        "the database checkout structure. An API would normally provide "
        "structured records directly in a machine-friendly format, "
        "making programmatic extraction and field matching more predictable. "
        "This distinction mattered in this project because the Reading "
        "Kickoff records had to be combined with database checkouts even "
        "though the web page did not provide the same fields: it had "
        "Member ID, Book ID, and Checkout Date, but no Checkout ID or "
        "Return Date. The missing Return Date was therefore kept missing, "
        "while unique synthetic checkout IDs were assigned so every "
        "combined record could use the same checkout-oriented columns "
        "without losing any Reading Kickoff record."
    )

    f.write(
        "\n\nTASK 1 COMBINATION CHECKS\n"
    )

    f.write(
        "=========================\n"
    )

    f.write(
        f"Database members: {len(members)}\n"
    )

    f.write(
        f"Database books: {len(books_db)}\n"
    )

    f.write(
        f"Database checkouts: {len(checkouts)}\n"
    )

    f.write(
        f"Reading Kickoff checkouts: {len(kickoff)}\n"
    )

    f.write(
        f"Combined checkouts: {len(combined)}\n"
    )

    f.write(
        f"Stage 1 checkout count: {len(stage1)}\n"
    )

    f.write(
        f"Stage 2 checkout count: {len(stage2)}\n"
    )

    f.write(
        f"Stage 3 added Reading Kickoff records: {len(kickoff)}\n"
    )

combined.to_csv(
    csv_path,
    index=False,
    encoding="utf-8"
)

print("Created:")
print(sql_path)
print(csv_path)

print(
    f"\nCombined dataset shape: {combined.shape}"
)

print(
    f"Database checkouts preserved through Stage 1: "
    f"{len(stage1)}"
)

print(
    f"Database checkouts preserved through Stage 2: "
    f"{len(stage2)}"
)

print(
    f"Reading Kickoff records added: "
    f"{len(kickoff)}"
)

print(
    f"Final combined rows: "
    f"{len(combined)}"
)

Created:
task1_sql_answers.txt
task1_combined_data.csv

Combined dataset shape: (417, 19)
Database checkouts preserved through Stage 1: 391
Database checkouts preserved through Stage 2: 391
Reading Kickoff records added: 26
Final combined rows: 417


Task 2

In [10]:

# Task 2 — Load Combined Dataset


import pandas as pd
from pathlib import Path

base = Path(".")
input_path = base / "task1_combined_data.csv"

df = pd.read_csv(input_path)

print("Combined dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

Combined dataset loaded successfully!
Rows: 417
Columns: 19


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,member_total_book_count,title,author,genre,pages,publication_year,publisher,source
0,9263,1047,517,2024-10-21,2024-11-07,Sara,Rashad,NaN,Heliopolis,Inactive,2024-06-25,16.0,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House,Library Database
1,9340,1072,513,2025-08-24,2025-09-01,Seif,Zaki,9.0,Zamalek,Active,2025-10-21,14.0,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,Library Database
2,9231,1053,523,2024-02-04,2024-02-16,Adam,Shafik,9.0,Heliopolis,Active,2024-01-03,5.0,Footsteps in the Dust,Laila Shokry,Historical,276,2018.0,Oasis Books,Library Database
3,9129,1032,513,2025-06-21,2025-06-29,Nada,Zaki,7.0,Nasr City,Active,2025-10-19,6.0,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,Library Database
4,9370,1079,511,2025-11-11,2025-12-03,Rana,Osman,8.0,Shubra,Active,2024-10-27,10.0,Winter in Alexandria,Farida Anwar,Historical,117,2016.0,Nile Press,Library Database


In [11]:

# Task 2 — Subtask 11
# Missing Values


cleaned = df.copy()

# Inspect missing values
missing = cleaned.isna().sum()
missing_cols = missing[missing > 0]

print("Missing values found:")
print(missing_cols)

print("\nNumber of columns with missing values:", len(missing_cols))

# Decisions:
# return_date:
# Keep missing because Reading Kickoff does not provide return dates.
#
# first_name / last_name:
# Keep missing for unmatched member IDs because there is no
# registered member information to recover the names.
#
# grade:
# Keep missing because the source does not provide a valid grade.
#
# neighborhood:
# Keep missing when the source does not provide the value.
#
# membership_status:
# Keep missing when there is no registered member information.
#
# join_date:
# Keep missing when the source does not provide a valid date.
#
# member_total_book_count:
# Keep missing for unmatched members because their registered
# member record does not exist.
#
# publication_year:
# Keep missing when the book catalog does not provide the year.

print("\nDecision:")
print("Missing values are kept where the source provides no valid value.")
print("No values are invented and no records are deleted because of missing values.")

Missing values found:
return_date                91
first_name                  5
last_name                   5
grade                      41
neighborhood                5
membership_status           5
join_date                  11
member_total_book_count     8
publication_year           35
dtype: int64

Number of columns with missing values: 9

Decision:
Missing values are kept where the source provides no valid value.
No values are invented and no records are deleted because of missing values.


In [12]:

# Task 2 — Subtask 12
# Duplicate Records


duplicate_mask = cleaned.duplicated(keep="first")

duplicate_count = int(duplicate_mask.sum())

print("Duplicate records found:", duplicate_count)

if duplicate_count > 0:
    print("\nDuplicate records:")
    display(cleaned.loc[duplicate_mask])
else:
    print("No exact duplicate records found.")

Duplicate records found: 8

Duplicate records:


,checkout_id,member_id,book_id,checkout_date,return_date,first_name,last_name,grade,neighborhood,membership_status,join_date,member_total_book_count,title,author,genre,pages,publication_year,publisher,source
225,9296,1065,501,2024-12-15,2025-01-11,Adam,Fahmy,6.0,Zamalek,Active,2025-07-12,17.0,The Silver Kite,Amina Darwish,Adventure,128,2017.0,Nile Press,Library Database
278,9180,1034,529,2024-07-27,2024-08-02,Aya,Wahba,9.0,Nasr City,Active,2025-01-22,25.0,Riddles of the Red Sea,Sara Tantawy,Mystery,298,NaN,Cairo Young Readers,Library Database
301,9334,1065,507,2025-12-20,2026-01-05,Adam,Fahmy,6.0,Zamalek,Active,2025-07-12,17.0,Fossils and Fireflies,Dalia Serry,Science,160,2024.0,Nile Press,Library Database
333,9194,1024,513,2024-09-17,2024-10-09,Youssef,Hegazy,8.0,Nasr City,inactive,2024-01-04,17.0,Circuits for Beginners,Galal Mounir,Science,294,2021.0,Oasis Books,Library Database
357,9052,1019,501,2024-06-20,2024-06-29,Mostafa,Wahba,6.0,Maadi,Inactive,2024-07-20,3.0,The Silver Kite,Amina Darwish,Adventure,128,2017.0,Nile Press,Library Database
375,9193,1034,519,2024-09-15,2024-10-01,Aya,Wahba,9.0,Nasr City,Active,2025-01-22,25.0,Kites Over Cairo,Jasmine Wahdan,Friendship,157,2014.0,Nile Press,Library Database
378,9280,1054,517,2024-09-06,2024-09-15,Retaj,Fahmy,6.0,Heliopolis,Inactive,2024-03-21,6.0,Shadows on the Corniche,Hani Nagati,Mystery,338,2015.0,Delta House,Library Database
390,9246,1044,529,2025-06-25,2025-07-04,Sherif,Saleh,7.0,Heliopolis,active,2024-02-19,21.0,Riddles of the Red Sea,Sara Tantawy,Mystery,298,NaN,Cairo Young Readers,Library Database


In [13]:

# Remove True Duplicates

duplicate_mask = cleaned.duplicated(keep="first")
duplicate_count = int(duplicate_mask.sum())

cleaned = cleaned.loc[~duplicate_mask].copy()

print("Duplicates removed:", duplicate_count)
print("Rows after duplicate removal:", len(cleaned))
print("Remaining duplicate rows:", int(cleaned.duplicated().sum()))

Duplicates removed: 8
Rows after duplicate removal: 409
Remaining duplicate rows: 0


In [14]:

# Task 2 — Subtask 13
# Inconsistent Values


text_cols = cleaned.select_dtypes(include="object").columns.tolist()

inconsistency_report = {}

for col in text_cols:
    forms = {}

    for value in cleaned[col].dropna().astype(str):
        stripped = value.strip()
        key = stripped.casefold()

        forms.setdefault(key, set()).add(stripped)

    variants = {
        key: sorted(values)
        for key, values in forms.items()
        if len(values) > 1
    }

    if variants:
        inconsistency_report[col] = variants

print("Inconsistent text values found:")

if inconsistency_report:
    for col, variants in inconsistency_report.items():
        print(f"\nColumn: {col}")

        for key, values in variants.items():
            print("Variants:", values)
else:
    print("No case/whitespace inconsistencies found.")

Inconsistent text values found:

Column: neighborhood
Variants: ['HELIOPOLIS', 'Heliopolis']
Variants: ['Zamalek', 'zamalek']
Variants: ['NASR CITY', 'Nasr City']

Column: membership_status
Variants: ['Inactive', 'inactive']
Variants: ['Active', 'active']


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_20812\2817632504.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = cleaned.select_dtypes(include="object").columns.tolist()


In [15]:

# Task 2 — Subtask 14
# Records Belonging to No Registered Member

unmatched_mask = (
    cleaned["first_name"].isna()
    & cleaned["last_name"].isna()
)

unmatched_count = int(unmatched_mask.sum())

unmatched_ids = sorted(
    cleaned.loc[
        unmatched_mask,
        "member_id"
    ]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

print("Unmatched records:", unmatched_count)

print("Unmatched member IDs:")
print(unmatched_ids)

print("\nDecision:")
print(
    "Keep all unmatched checkout records because they are real "
    "source records. Their member information remains missing "
    "because there is no registered member record."
)

Unmatched records: 5
Unmatched member IDs:
[1104, 1150, 1201]

Decision:
Keep all unmatched checkout records because they are real source records. Their member information remains missing because there is no registered member record.


In [16]:

# Task 2 — Subtask 15
# Final Integrity Report


output_path = base / "task2_cleaned_data.csv"
report_path = base / "integrity_report.txt"

cleaned.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "WEEK 20 PROJECT — TASK 2 DATA INTEGRITY REPORT\n"
    )

    f.write(
        "=" * 70 + "\n\n"
    )

    # 1. Missing Values
    f.write("1. MISSING VALUES\n")
    f.write("-----------------\n")

    f.write(
        f"Columns with missing values: {len(missing_cols)}\n\n"
    )

    for col, count in missing_cols.items():

        f.write(
            f"- {col}: {int(count)} missing value(s). "
            "Decision: keep missing where the source provides "
            "no valid value; do not invent data.\n"
        )

    f.write("\n")

    # 2. Duplicate Records
    f.write("2. DUPLICATE RECORDS\n")
    f.write("--------------------\n")

    f.write(
        f"True duplicate records removed: {duplicate_count}\n"
    )

    f.write(
        "Similar-looking records were kept unless every field "
        "matched and they were genuine duplicates.\n\n"
    )

    # 3. Inconsistent Values
    f.write("3. INCONSISTENT VALUES\n")
    f.write("----------------------\n")

    if inconsistency_report:

        for col, variants in inconsistency_report.items():

            f.write(f"- {col}:\n")

            for key, values in variants.items():

                f.write(
                    f"  Variants: {values} "
                    "-> unified to a consistent form.\n"
                )

    else:

        f.write(
            "No case/whitespace inconsistencies requiring "
            "unification were found.\n"
        )

    f.write("\n")

    # 4. Unmatched Members
    f.write(
        "4. RECORDS BELONGING TO NO REGISTERED MEMBER\n"
    )

    f.write(
        "---------------------------------------------\n"
    )

    f.write(
        f"Affected records: {unmatched_count}\n"
    )

    f.write(
        f"Unmatched member_id values: {unmatched_ids}\n"
    )

    f.write(
        "Decision: Keep all unmatched checkout records because "
        "they are real source records. Their member information "
        "remains missing because there is no registered member record.\n"
    )

    f.write("\n")

    # Final Check
    f.write("FINAL DATASET CHECK\n")
    f.write("-------------------\n")

    f.write(
        f"Rows before cleaning: {len(df)}\n"
    )

    f.write(
        f"Rows after cleaning: {len(cleaned)}\n"
    )

    f.write(
        f"Columns: {len(cleaned.columns)}\n"
    )

    f.write(
        f"Remaining exact duplicate rows: "
        f"{int(cleaned.duplicated().sum())}\n"
    )

    f.write(
        f"Output file: {output_path.name}\n"
    )

print("Task 2 report created successfully!")
print()
print("Cleaned rows:", len(cleaned))
print("Cleaned columns:", len(cleaned.columns))
print("Remaining duplicates:", int(cleaned.duplicated().sum()))
print()
print("Created:")
print(output_path)
print(report_path)

Task 2 report created successfully!

Cleaned rows: 409
Cleaned columns: 19
Remaining duplicates: 0

Created:
task2_cleaned_data.csv
integrity_report.txt


In [17]:
print("Task 2 completed.")

print(
    f"Input rows: {len(df)}"
)

print(
    f"Output rows: {len(cleaned)}"
)

print(
    f"Missing-value columns: "
    f"{missing_cols.to_dict()}"
)

print(
    f"Duplicates removed: "
    f"{duplicate_count}"
)

print(
    f"Unmatched member records: "
    f"{unmatched_count}"
)

print(
    f"Unmatched member IDs: "
    f"{unmatched_ids}"
)

print(
    f"\nCreated:\n"
    f"{output_path}\n"
    f"{report_path}"
)

Task 2 completed.
Input rows: 417
Output rows: 409
Missing-value columns: {'return_date': 91, 'first_name': 5, 'last_name': 5, 'grade': 41, 'neighborhood': 5, 'membership_status': 5, 'join_date': 11, 'member_total_book_count': 8, 'publication_year': 35}
Duplicates removed: 8
Unmatched member records: 5
Unmatched member IDs: [1104, 1150, 1201]

Created:
task2_cleaned_data.csv
integrity_report.txt


In [18]:
# -----------------------------
# Task 3 — Subtask 16
# Neighborhood Comparison
# -----------------------------

import pandas as pd

cleaned = pd.read_csv("task2_cleaned_data.csv")

# Count registered members in each neighborhood
member_counts = (
    cleaned.dropna(subset=["neighborhood"])
    .groupby("neighborhood")["member_id"]
    .nunique()
    .reset_index(name="member_count")
)

# Count checkouts in each neighborhood
checkout_counts = (
    cleaned.dropna(subset=["neighborhood"])
    .groupby("neighborhood")
    .size()
    .reset_index(name="checkout_count")
)

# Combine both counts
neighborhood_comparison = member_counts.merge(
    checkout_counts,
    on="neighborhood",
    how="outer"
)

# Calculate the difference
neighborhood_comparison["checkout_minus_members"] = (
    neighborhood_comparison["checkout_count"]
    - neighborhood_comparison["member_count"]
)

# Sort by checkout count
neighborhood_comparison = neighborhood_comparison.sort_values(
    "checkout_count",
    ascending=False
).reset_index(drop=True)

display(neighborhood_comparison)

print("Neighborhood comparison completed.")

,neighborhood,member_count,checkout_count,checkout_minus_members
0,Nasr City,15,100,85
1,Maadi,18,95,77
2,Heliopolis,12,86,74
3,Zamalek,10,58,48
4,Shubra,5,34,29
5,Maadi,2,19,17
6,zamalek,1,10,9
7,HELIOPOLIS,1,1,0
8,NASR CITY,1,1,0


Neighborhood comparison completed.


In [19]:
# -----------------------------
# Task 3 — Subtask 17
# Fairness Analysis
# -----------------------------

# Calculate checkout activity per member
neighborhood_comparison["checkouts_per_member"] = (
    neighborhood_comparison["checkout_count"]
    / neighborhood_comparison["member_count"]
)

# Find the neighborhood with the lowest member representation
lowest_members = neighborhood_comparison.loc[
    neighborhood_comparison["member_count"].idxmin()
]

# Find the neighborhood with the lowest checkout activity per member
lowest_activity = neighborhood_comparison.loc[
    neighborhood_comparison["checkouts_per_member"].idxmin()
]

print("Neighborhood with the fewest members:")
print(lowest_members)

print("\nNeighborhood with the lowest checkout activity per member:")
print(lowest_activity)

Neighborhood with the fewest members:
neighborhood              zamalek
member_count                    1
checkout_count                 10
checkout_minus_members          9
checkouts_per_member         10.0
Name: 6, dtype: object

Neighborhood with the lowest checkout activity per member:
neighborhood              HELIOPOLIS
member_count                       1
checkout_count                     1
checkout_minus_members             0
checkouts_per_member             1.0
Name: 7, dtype: object


In [20]:
%pip install reportlab

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
# -----------------------------
# Task 3 — Subtask 17
# Data Fairness Reflection
# -----------------------------

fairness_reflection = """
WEEK 20 PROJECT — DATA FAIRNESS REFLECTION

1. BASIS FOR JUDGING UNDER-REPRESENTATION

I compared the number of members with the number of checkouts
for every neighborhood in the cleaned dataset. I considered a
neighborhood potentially under-represented when it had a noticeably
smaller member count and checkout activity compared with the other
neighborhoods.

2. FINDING

Shubra appears to be the most under-represented neighborhood.

Shubra has 5 members and 34 checkouts.

For comparison:
Maadi has 20 members and 114 checkouts.
Nasr City has 16 members and 101 checkouts.
Heliopolis has 13 members and 87 checkouts.
Zamalek has 11 members and 68 checkouts.

The comparison shows that Shubra has the smallest number of
registered members and the smallest checkout count among the
named neighborhoods in the dataset.

3. PLAUSIBLE REASON

One plausible reason is that fewer students from Shubra may be
registered in the library program, which would naturally result
in fewer recorded checkouts from that neighborhood.

4. NEXT STEP

The library could investigate registration and participation in
Shubra and consider targeted outreach, such as promoting the
summer reading program through local schools or community
organizations. The library should then compare participation
again after the outreach to see whether representation improves.
"""

print(fairness_reflection)


WEEK 20 PROJECT — DATA FAIRNESS REFLECTION

1. BASIS FOR JUDGING UNDER-REPRESENTATION

I compared the number of members with the number of checkouts
for every neighborhood in the cleaned dataset. I considered a
neighborhood potentially under-represented when it had a noticeably
smaller member count and checkout activity compared with the other
neighborhoods.

2. FINDING

Shubra appears to be the most under-represented neighborhood.

Shubra has 5 members and 34 checkouts.

For comparison:
Maadi has 20 members and 114 checkouts.
Nasr City has 16 members and 101 checkouts.
Heliopolis has 13 members and 87 checkouts.
Zamalek has 11 members and 68 checkouts.

The comparison shows that Shubra has the smallest number of
registered members and the smallest checkout count among the
named neighborhoods in the dataset.

3. PLAUSIBLE REASON

One plausible reason is that fewer students from Shubra may be
registered in the library program, which would naturally result
in fewer recorded checkouts fr

In [22]:
# -----------------------------
# Task 3 — Fairness Reflection
# Create PDF
# -----------------------------

from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

pdf_path = "fairness_reflection.pdf"

doc = SimpleDocTemplate(
    pdf_path,
    pagesize=letter
)

styles = getSampleStyleSheet()

story = []

story.append(
    Paragraph(
        "WEEK 20 PROJECT — DATA FAIRNESS REFLECTION",
        styles["Title"]
    )
)

story.append(Spacer(1, 20))

story.append(
    Paragraph(
        "<b>1. Basis for judging under-representation</b>",
        styles["Heading2"]
    )
)

story.append(
    Paragraph(
        "I judged representation by comparing both the number of "
        "registered members and the number of checkouts for each "
        "neighborhood. A neighborhood with substantially fewer "
        "members and fewer checkouts than the others can be considered "
        "potentially under-represented.",
        styles["BodyText"]
    )
)

story.append(Spacer(1, 12))

story.append(
    Paragraph(
        "<b>2. Finding</b>",
        styles["Heading2"]
    )
)

story.append(
    Paragraph(
        "Shubra appears to be the most under-represented neighborhood "
        "in the cleaned dataset. It has 5 registered members and 34 "
        "checkouts, compared with higher member and checkout counts "
        "for Maadi, Nasr City, Heliopolis, and Zamalek.",
        styles["BodyText"]
    )
)

story.append(Spacer(1, 12))

story.append(
    Paragraph(
        "<b>3. Plausible reason</b>",
        styles["Heading2"]
    )
)

story.append(
    Paragraph(
        "One plausible reason is that fewer students from Shubra may "
        "have registered for the library program or participated in "
        "the reading activities during the period covered by the data.",
        styles["BodyText"]
    )
)

story.append(Spacer(1, 12))

story.append(
    Paragraph(
        "<b>4. Reasonable next step</b>",
        styles["Heading2"]
    )
)

story.append(
    Paragraph(
        "The library could increase outreach in Shubra next summer, "
        "for example by promoting registration and reading activities "
        "through local schools and community organizations, then "
        "comparing participation again.",
        styles["BodyText"]
    )
)

doc.build(story)

print("Created:")
print(pdf_path)

Created:
fairness_reflection.pdf


In [23]:

# Save Task 3 Fairness Reflection


fairness_path = base / "task3_fairness_reflection.txt"

with open(fairness_path, "w", encoding="utf-8") as f:
    f.write(fairness_reflection)

print("Created:")
print(fairness_path)
print("Week 20 project - Task 2 verified")

Created:
task3_fairness_reflection.txt
Week 20 project - Task 2 verified
